In [ ]:
# For removing unwanted columns from the dataframe!

In [ ]:
Purpose: this file is used to remove unwanted columns from the csv file and then make it to parquet file.

In [4]:
import pandas as pd

data = pd.read_csv("../dataset/xl/xl.csv")
df = pd.DataFrame(data)
print(df.columns)

# usefull - id, name, album, artists, energy, danceability, valence, tempo, year, loudness, cover_url

Index(['id', 'name', 'album', 'album_id', 'artists', 'artist_ids',
       'track_number', 'disc_number', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms',
       'time_signature', 'year', 'release_date', 'cover_url'],
      dtype='object')


In [5]:
cols_to_drop = df.columns.difference(['id', 'name', 'album', 'artists', 'energy', 'danceability', 'valence', 'tempo', 'year', 'loudness', 'cover_url'])
df.drop(columns=cols_to_drop, inplace=True)

In [6]:
print(df.columns)
df.to_csv("../dataset/xl/xl_cleaned.csv", index=False)
df.to_parquet("../dataset/xl/xl.parquet", engine="pyarrow", compression="snappy")

Index(['id', 'name', 'album', 'artists', 'danceability', 'energy', 'loudness',
       'valence', 'tempo', 'year', 'cover_url'],
      dtype='object')


In [19]:
import gc

# Delete the large object
del df 
gc.collect()

0

In [1]:
# New Approach - please use this for now, this also optimizes the columns like downcasting floats and ints
# In case this fails, above 3 cells will do the job (except downcasting).
import pandas as pd

df = pd.read_csv("../dataset/xl/xl.csv")

cols_to_keep = ['id', 'name', 'album', 'artists', 'energy', 'danceability', 'valence', 'tempo', 'year', 'loudness', 'cover_url']
df = df[cols_to_keep]

# -----------------------------
# Downcast float64 → float32
# -----------------------------
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

# -----------------------------
# Downcast int64 → smaller int
# -----------------------------
df["year"] = pd.to_numeric(df["year"], downcast="integer")

# -----------------------------
# Converting repeated strings to category
# -----------------------------
df["album"] = df["album"].astype("category")
df["artists"] = df["artists"].astype("category")

df.to_parquet("../dataset/xl/downcasted_xl.parquet", engine="pyarrow", compression="zstd")

print("Final memory usage:", df.memory_usage(deep=True).sum() / 1024**2, "MB")

Final memory usage: 411.8877182006836 MB
